# 03 - Métricas de Evaluación, Matriz de Confusión y Curvas ROC / PR

## Curso de Machine Learning: Modelos de Clasificación

En este cuaderno estudiaremos el marco riguroso de evaluacion para modelos de clasificacion. Aprenderemos por que la exactitud (accuracy) puede ser una metrica peligrosamente enganosa en presencia de clases desbalanceadas o costos asimetricos de error, desglosaremos la **matriz de confusion**, deduciremos las formulas de **precision**, **recall (sensibilidad)**, **especificidad** y **F1-Score**, analizaremos la calibracion del umbral de decision clinico, y trazaremos las curvas **ROC** y **Precision-Recall**.

---

### Contenidos
1. La Falacia de la Exactitud (Accuracy Paradox)
2. La Matriz de Confusion: TP, TN, FP y FN
3. Metricas Fundamentales: Precision, Recall, Especificidad y F1-Score
4. El Dilema Clinico: Errores Tipo I vs. Errores Tipo II
5. Evaluacion con Todas las Variables en `load_breast_cancer`
6. Calibracion y Sensibilidad del Umbral de Decision
7. Curva ROC y Area Bajo la Curva (ROC-AUC)
8. Curva Precision-Recall y Precision Promedio (Average Precision)
9. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
    precision_score, recall_score, f1_score, accuracy_score,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. La Falacia de la Exactitud (Accuracy Paradox)

La **Exactitud (Accuracy)** mide la fraccion total de predicciones correctas:

$$\text{Accuracy} = \frac{\text{Aciertos Totales}}{\text{Muestras Totales}} = \frac{TP + TN}{TP + TN + FP + FN}$$

### ¿Por que falla?
Si en un problema medico de deteccion temprana de una enfermedad rara solo el $1\%$ de los pacientes esta enfermo, un clasificador trivial que prediga ciegamente "Sano" para el $100\%$ de las personas alcanzara una exactitud del $99\%$. Sin embargo, este modelo es completamente inutil y peligroso porque no detecta a ningun paciente enfermo.


## 2. La Matriz de Confusión: TP, TN, FP y FN

La matriz de confusion contrasta las etiquetas reales contra las predicciones del estimador:

| | Predicho Negativo (0) | Predicho Positivo (1) |
| :--- | :---: | :---: |
| **Real Negativo (0)** | **Verdadero Negativo (TN)** | **Falso Positivo (FP)** *(Error Tipo I)* |
| **Real Positivo (1)** | **Falso Negativo (FN)** *(Error Tipo II)* | **Verdadero Positivo (TP)** |

### En el Contexto Medico de Cancer de Mama:
- **Falso Positivo (FP)**: Un paciente con tumor benigno es clasificado como maligno. Conlleva ansiedad y pruebas confirmatorias invasivas adicionales.
- **Falso Negativo (FN)**: Un paciente con tumor maligno canceroso es clasificado como benigno y enviado a casa sin tratamiento. **Este es el error mas critico y potencialmente letal.**


## 3. Métricas Fundamentales Derivadas

### Precision (Valor Predictivo Positivo)
De todas las observaciones que el modelo clasifico como positivas, ¿que proporcion fue verdaderamente positiva?
$$\text{Precision} = \frac{TP}{TP + FP}$$
*Prioridad*: Minimizar falsas alarmas (ej. filtros de spam o recomendaciones de compras).

### Recall / Sensibilidad (True Positive Rate)
De todas las observaciones verdaderamente positivas en la realidad, ¿que proporcion logro capturar el modelo?
$$\text{Recall} = \frac{TP}{TP + FN}$$
*Prioridad*: No dejar escapar ningun positivo real (ej. diagnosticos medicos graves, deteccion de fraudes).

### Especificidad (True Negative Rate)
De todas las observaciones reales negativas, ¿que proporcion fue identificada correctamente como negativa?
$$\text{Especificidad} = \frac{TN}{TN + FP}$$

### F1-Score
Media armonica ponderada entre Precision y Recall. Penaliza severamente si alguna de las dos metricas es muy baja:
$$\text{F1-Score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 TP}{2 TP + FP + FN}$$


## 4. Entrenamiento con las 30 Variables en `load_breast_cancer`

Ajustaremos un modelo completo utilizando las 30 caracteristicas clinicas estandarizadas mediante un `Pipeline`.


In [ ]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.frame.drop(columns=['target'])
y = cancer.frame['target'] # Recordatorio: 0 = Maligno, 1 = Benigno

# En medicina suele definirse la condicion de interes (Maligno) como clase positiva.
# Crearemos y_maligno donde 1 = Maligno y 0 = Benigno para un analisis clinico intuitivo
y_maligno = (y == 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_maligno, test_size=0.20, random_state=42, stratify=y_maligno
)

# Pipeline estandar
pipeline_completo = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, max_iter=1000))
])

pipeline_completo.fit(X_train, y_train)

# Predicciones iniciales con umbral estandar (0.5)
y_pred = pipeline_completo.predict(X_test)
y_probs = pipeline_completo.predict_proba(X_test)[:, 1] # Probabilidad de Maligno


In [ ]:
# Matriz de Confusion
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benigno (0)', 'Maligno (1)'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Matriz de Confusion (Umbral = 0.50)', fontsize=12, fontweight='bold')
plt.grid(False)
plt.show()

# Reporte detallado de clasificacion
print("Reporte Completo de Clasificacion:")
print(classification_report(y_test, y_pred, target_names=['Benigno', 'Maligno'], digits=4))


## 5. Calibración y Sensibilidad del Umbral de Decisión

Por defecto, Scikit-Learn asigna clase positiva cuando $P(Y=1 \mid X) \ge 0.5$. Sin embargo, en un entorno de oncologia, es preferible aceptar algunos Falsos Positivos adicionales a cambio de reducir a **cero** los Falsos Negativos.

Variemos el umbral de decision $T \in [0.1, 0.3, 0.5, 0.7, 0.9]$ y observemos el impacto directo en la matriz de confusion, Precision y Recall.


In [ ]:
umbrales = [0.10, 0.25, 0.50, 0.75, 0.90]
resultados_umbrales = []

for u in umbrales:
    y_pred_u = (y_probs >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_u).ravel()
    prec = precision_score(y_test, y_pred_u, zero_division=0)
    rec = recall_score(y_test, y_pred_u)
    f1 = f1_score(y_test, y_pred_u)
    
    resultados_umbrales.append({
        'Umbral': u,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
        'Recall (Sensibilidad)': rec,
        'Precision': prec,
        'F1-Score': f1
    })

df_umbrales = pd.DataFrame(resultados_umbrales)
print("Impacto de la Variacion del Umbral de Decision en Diagnostico:")
print(df_umbrales.round(4).to_string(index=False))


## 6. Curva ROC y Área Bajo la Curva (ROC-AUC)

La curva **ROC (Receiver Operating Characteristic)** evalua el clasificador a traves de **todos los posibles umbrales de decision**:
- **Eje Y**: Tasa de Verdaderos Positivos (TPR o Recall): $\frac{TP}{TP + FN}$
- **Eje X**: Tasa de Falsos Positivos (FPR): $\frac{FP}{FP + TN} = 1 - \text{Especificidad}$

### Metrica ROC-AUC:
- $\text{AUC} = 1.0$: Clasificador perfecto con separacion absoluta entre clases.
- $\text{AUC} = 0.5$: Clasificador aleatorio sin capacidad de discriminacion (linea diagonal discontinua).
- $\text{AUC} > 0.90$: Rendimiento excelente en aplicaciones clinicas.


In [ ]:
fpr, tpr, thresholds_roc = roc_curve(y_test, y_probs)
auc_score = roc_auc_score(y_test, y_probs)

precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_probs)
ap_score = average_precision_score(y_test, y_probs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# 1. Curva ROC
axes[0].plot(fpr, tpr, color='#1f77b4', linewidth=2.5, label=f'Curva ROC (AUC = {auc_score:.4f})')
axes[0].plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Clasificador Aleatorio (AUC = 0.50)')
axes[0].set_title('Curva ROC (Receiver Operating Characteristic)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Tasa de Falsos Positivos (1 - Especificidad)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Tasa de Verdaderos Positivos (Recall)', fontsize=11, fontweight='bold')
axes[0].legend(loc='lower right', fontsize=10)

# 2. Curva Precision-Recall
axes[1].plot(recalls, precisions, color='#2ca02c', linewidth=2.5, label=f'Curva PR (AP = {ap_score:.4f})')
axes[1].axhline(np.mean(y_test), color='gray', linestyle='--', linewidth=1.5, label=f'Linea Base ({np.mean(y_test):.2f})')
axes[1].set_title('Curva Precision-Recall', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Recall (Sensibilidad)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Precision', fontsize=11, fontweight='bold')
axes[1].legend(loc='lower left', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Metricas Globales de Capacidad Discriminante:")
print(f"  Area Bajo la Curva ROC (ROC-AUC):               {auc_score:.4f}")
print(f"  Precision Promedio (Average Precision / PR-AUC): {ap_score:.4f}")


## 7. Conclusiones

### Puntos Clave Aprendidos:
1. **La exactitud no basta**: En aplicaciones criticas como oncologia o fraude financiero, un modelo con alta exactitud puede ser peligroso si omite la clase minoritaria o mas riesgosa.
2. **Precision vs. Recall**: Existe un compromiso inherente (tradeoff) entre ambas metricas gobernado por el umbral de decision.
3. **Calibracion del Umbral**: Ajustar el umbral $T$ por debajo de $0.5$ permite priorizar el Recall y minimizar los Falsos Negativos en diagnosticos de salud.
4. **ROC-AUC y PR-AUC**: Evaluan la calidad de ordenamiento y discriminacion probabilistica del modelo de forma independiente a la eleccion de un umbral rigido particular.

En el siguiente cuaderno, **04_Regularizacion_en_Regresion_Logistica_y_Multiclase.ipynb**, incorporaremos penalizaciones $L_1$ y $L_2$ para controlar el sobreajuste y seleccionar variables clinicas en clasificacion, y generalizaremos el algoritmo a escenarios **multiclase** con la formulacion Softmax.
